# Noise-diversity study — Colab GPU run notebook

Runs the full protocol-v1 matrix (untuned baseline + 18 conditions × 3 seeds = 55 runs) on a Colab GPU.

* **Resumable**: every finished run writes `results/runs/<run_id>/metrics.json` to Google Drive.
  If the session disconnects, open the notebook again and **Run all**. Finished runs are skipped.
* Expected: roughly 5–10 min per run on a T4, so it takes a few sessions.
* When everything is done, run the last cell and download `results_export.zip`.

**Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# ---- settings -------------------------------------------------------------
REPO_URL = "https://github.com/SoumyaranjanPanda01/noise-diversity-research.git"
BRANCH = "main"
DRIVE_RESULTS = "/content/drive/MyDrive/noise-diversity-results"
SEEDS = "0 1 2"
# If the repo is private: add a Colab secret named GITHUB_TOKEN (key icon in the left bar).

In [ ]:
import os, subprocess

url = REPO_URL
try:  # private repo: Colab secret GITHUB_TOKEN (key icon in the left bar)
    from google.colab import userdata
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass  # public repo / no secret

os.chdir("/content")
if not os.path.exists("noise-diversity-research"):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "noise-diversity-research"], check=True)
os.chdir("noise-diversity-research")
subprocess.run(["git", "pull", "--ff-only"], check=False)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Pin the library versions used for development (recorded per run in run_config.json).
!pip -q install "transformers==5.21.0" "peft==0.21.2" "accelerate>=1.0" nltk matplotlib
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Switch the runtime to a GPU."

In [ ]:
# Keep results/ on Google Drive so nothing is lost when the session ends.
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_RESULTS, exist_ok=True)
if os.path.isdir("results") and not os.path.islink("results"):
    subprocess.run(f"cp -rn results/. {DRIVE_RESULTS}/ && rm -rf results", shell=True, check=True)
if not os.path.exists("results"):
    os.symlink(DRIVE_RESULTS, "results")
print(os.path.realpath("results"))

### 1. GPU smoke test (≈1 min) — checks the fp16/bf16 training path end-to-end

In [ ]:
!python scripts/train.py --condition fold_a/mixed3_f0.50 --seed 0 --max-steps 5 --eval-limit 32 --tag gpusmoke --no-save-adapter --force

### 2. Plan / status

In [ ]:
!python scripts/run_all.py --seeds {SEEDS} --dry-run

### 3. Run the matrix (resumable — just re-run this cell in a new session)
Order: untuned → primary comparison (clean, singles, Mixed-3) → sweeps → NEFTune/EDA baselines.
Per-run stdout is in `results/runs/<run_id>/stdout.log`.

In [ ]:
!python scripts/run_all.py --seeds {SEEDS}

### 4. Analysis + export

In [ ]:
!python scripts/analyze.py
!cd results && zip -qr ../results_export.zip runs analysis run_registry.csv -x "checkpoints/*"
print("download:", os.path.abspath("results_export.zip"))